In [1]:
!pip install google-api-python-client python-dotenv pandas

In [2]:
import os
import pandas as pd
from dotenv import load_dotenv
from googleapiclient.discovery import build

In [3]:
load_dotenv("../.env")  # since notebook is inside src

API_KEY = os.getenv("YOUTUBE_API_KEY")

if API_KEY:
    print("API Key loaded successfully")
else:
    print("API Key not found")

API Key loaded successfully


In [4]:
youtube = build("youtube", "v3", developerKey=API_KEY)

print("YouTube API Connected Successfully")

YouTube API Connected Successfully


In [5]:
load_dotenv("../.env")  # since notebook is inside src

API_KEY = os.getenv("YOUTUBE_API_KEY")

if API_KEY:
    print("API Key loaded successfully")
else:
    print("API Key not found")

API Key loaded successfully


In [6]:
# Step 1: Get uploads playlist ID

channel_id = "UC8butISFwT-Wl7EV0hUK0BQ"  # FreeCodeCamp

channel_request = youtube.channels().list(
    part="contentDetails",
    id=channel_id
)

channel_response = channel_request.execute()

uploads_playlist_id = channel_response["items"][0]["contentDetails"]["relatedPlaylists"]["uploads"]

print("Uploads Playlist ID:", uploads_playlist_id)

Uploads Playlist ID: UU8butISFwT-Wl7EV0hUK0BQ


In [7]:
def get_all_videos_from_playlist(playlist_id):
    videos = []
    next_page_token = None

    while True:
        request = youtube.playlistItems().list(
            part="snippet",
            playlistId=playlist_id,
            maxResults=50,
            pageToken=next_page_token
        )

        response = request.execute()

        for item in response["items"]:
            videos.append(item)

        next_page_token = response.get("nextPageToken")

        if not next_page_token:
            break

    return videos

In [8]:
all_videos = get_all_videos_from_playlist(uploads_playlist_id)

print("Total videos fetched:", len(all_videos))

Total videos fetched: 2132


In [9]:
clean_videos = []

for item in all_videos:
    clean_videos.append({
        "video_id": item["snippet"]["resourceId"]["videoId"],
        "title": item["snippet"]["title"],
        "published_date": item["snippet"]["publishedAt"]
    })

In [10]:
import pandas as pd

df = pd.DataFrame(clean_videos)

df.head()

,video_id,title,published_date
0,4RGkpzTJjfg,Contributing to open source has many benefits ...,2026-02-25T13:29:55Z
1,UsfpzxZNsPo,Python Essentials for AI Agents – Tutorial,2026-02-25T11:00:52Z
2,U6-RekkuORI,Closures in JavaScript explained with a simple...,2026-02-24T13:05:28Z
3,bB5eA7vU9W8,Learn Notion – Full Course for Beginners,2026-02-24T11:00:57Z
4,Ai00MsNabaE,True or false: computers only understand 1s an...,2026-02-23T13:17:29Z


In [11]:
print("Total rows:", len(df))
print("Duplicate IDs:", df["video_id"].duplicated().sum())
print("Null values:\n", df.isnull().sum())
print("Shape:", df.shape)

Total rows: 2132
Duplicate IDs: 0
Null values:
 video_id          0
title             0
published_date    0
dtype: int64
Shape: (2132, 3)


In [12]:
df.to_csv("../data/raw_metadata.csv", index=False)

print("Milestone 1 completed successfully.")

Milestone 1 completed successfully.
